# EDA 01 - Schema and Quality Check

This notebook performs the first data-quality pass over the required CSV files: discovery, loading, shapes, schemas, missingness, duplicates, date detection, date ranges, and possible numeric columns stored as strings.

## 1. Setup and file discovery

Define the expected files, search recursively from the current working directory, and record any missing files without stopping execution.

In [1]:
import os
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 500)
pd.set_option("display.max_colwidth", 160)

PROJECT_ROOT = Path.cwd()
EXPECTED_FILES = [
    "products.csv",
    "customers.csv",
    "promotions.csv",
    "geography.csv",
    "orders.csv",
    "order_items.csv",
    "payments.csv",
    "shipments.csv",
    "returns.csv",
    "reviews.csv",
    "sales.csv",
    "inventory.csv",
    "web_traffic.csv",
    "sample_submission.csv",
]

DATE_NAME_PATTERN = re.compile(
    r"(date|time|timestamp|created|updated|order_date|ship_date|payment_date|return_date|review_date)",
    re.IGNORECASE,
)
DATE_PARSE_THRESHOLD = 0.80
NUMERIC_PARSE_THRESHOLD = 0.95
HIGH_MISSING_THRESHOLD = 50.0

all_csv_paths = list(PROJECT_ROOT.rglob("*.csv"))
paths_by_name = {}
for path in all_csv_paths:
    paths_by_name.setdefault(path.name, []).append(path)

records = []
for filename in EXPECTED_FILES:
    matches = paths_by_name.get(filename, [])
    if matches:
        selected_path = sorted(matches, key=lambda p: (len(p.parts), str(p)))[0]
        records.append({
            "file": filename,
            "status": "found",
            "path": str(selected_path.relative_to(PROJECT_ROOT)),
            "match_count": len(matches),
        })
    else:
        records.append({
            "file": filename,
            "status": "missing",
            "path": None,
            "match_count": 0,
        })

file_discovery_df = pd.DataFrame(records)
missing_files_df = file_discovery_df[file_discovery_df["status"] == "missing"].copy()

print(f"Project root: {PROJECT_ROOT}")
print(f"CSV files discovered recursively: {len(all_csv_paths)}")
file_discovery_df

Project root: D:\Code\Datathon 2026
CSV files discovered recursively: 15


,file,status,path,match_count
0,products.csv,found,datathon-2026-round-1\products.csv,1
1,customers.csv,found,datathon-2026-round-1\customers.csv,1
2,promotions.csv,found,datathon-2026-round-1\promotions.csv,1
3,geography.csv,found,datathon-2026-round-1\geography.csv,1
4,orders.csv,found,datathon-2026-round-1\orders.csv,1
5,order_items.csv,found,datathon-2026-round-1\order_items.csv,1
6,payments.csv,found,datathon-2026-round-1\payments.csv,1
7,shipments.csv,found,datathon-2026-round-1\shipments.csv,1
8,returns.csv,found,datathon-2026-round-1\returns.csv,1
9,reviews.csv,found,datathon-2026-round-1\reviews.csv,1


## 2. Load CSV files

Load each discovered CSV with `pandas.read_csv`. Files that fail to load are captured as warnings instead of crashing the notebook.

In [2]:
loaded = {}
load_records = []

for _, row in file_discovery_df.iterrows():
    filename = row["file"]
    if row["status"] != "found":
        load_records.append({
            "file": filename,
            "loaded": False,
            "rows": np.nan,
            "columns": np.nan,
            "error": "file missing",
        })
        continue

    path = PROJECT_ROOT / row["path"]
    try:
        df = pd.read_csv(path, low_memory=False)
        loaded[filename] = df
        load_records.append({
            "file": filename,
            "loaded": True,
            "rows": len(df),
            "columns": len(df.columns),
            "error": "",
        })
    except Exception as exc:
        load_records.append({
            "file": filename,
            "loaded": False,
            "rows": np.nan,
            "columns": np.nan,
            "error": repr(exc),
        })

load_summary_df = pd.DataFrame(load_records)
load_summary_df

,file,loaded,rows,columns,error
0,products.csv,True,2412,8,
1,customers.csv,True,121930,7,
2,promotions.csv,True,50,10,
3,geography.csv,True,39948,4,
4,orders.csv,True,646945,8,
5,order_items.csv,True,714669,7,
6,payments.csv,True,646945,4,
7,shipments.csv,True,566067,4,
8,returns.csv,True,39939,7,
9,reviews.csv,True,113551,7,


## 3. Dataset shape summary

Summarize row counts, column counts, and column names for every loaded dataset.

In [3]:
shape_rows = []
for filename, df in loaded.items():
    shape_rows.append({
        "file": filename,
        "rows": len(df),
        "columns": len(df.columns),
        "column_names": ", ".join(df.columns.astype(str)),
    })

shape_summary_df = pd.DataFrame(shape_rows).sort_values("file").reset_index(drop=True)
shape_summary_df

,file,rows,columns,column_names
0,customers.csv,121930,7,"customer_id, zip, city, signup_date, gender, age_group, acquisition_channel"
1,geography.csv,39948,4,"zip, city, region, district"
2,inventory.csv,60247,17,"snapshot_date, product_id, stock_on_hand, units_received, units_sold, stockout_days, days_of_supply, fill_rate, stockout_flag, overstock_flag, reorder_flag,..."
3,order_items.csv,714669,7,"order_id, product_id, quantity, unit_price, discount_amount, promo_id, promo_id_2"
4,orders.csv,646945,8,"order_id, order_date, customer_id, zip, order_status, payment_method, device_type, order_source"
5,payments.csv,646945,4,"order_id, payment_method, payment_value, installments"
6,products.csv,2412,8,"product_id, product_name, category, segment, size, color, price, cogs"
7,promotions.csv,50,10,"promo_id, promo_name, promo_type, discount_value, start_date, end_date, applicable_category, promo_channel, stackable_flag, min_order_value"
8,returns.csv,39939,7,"return_id, order_id, product_id, return_date, return_reason, return_quantity, refund_amount"
9,reviews.csv,113551,7,"review_id, order_id, product_id, customer_id, review_date, rating, review_title"


## 4. Schema and dtype summary

List every column with its pandas-inferred dtype and basic uniqueness information.

In [4]:
schema_rows = []
for filename, df in loaded.items():
    for col in df.columns:
        non_null = int(df[col].notna().sum())
        schema_rows.append({
            "file": filename,
            "column": col,
            "dtype": str(df[col].dtype),
            "non_null_count": non_null,
            "null_count": int(df[col].isna().sum()),
            "unique_non_null": int(df[col].nunique(dropna=True)),
            "sample_values": ", ".join(df[col].dropna().astype(str).head(5).tolist()),
        })

schema_summary_df = pd.DataFrame(schema_rows).sort_values(["file", "column"]).reset_index(drop=True)
schema_summary_df

,file,column,dtype,non_null_count,null_count,unique_non_null,sample_values
0,customers.csv,acquisition_channel,object,121930,0,6,"social_media, email_campaign, organic_search, referral, organic_search"
1,customers.csv,age_group,object,121930,0,5,"35-44, 45-54, 18-24, 35-44, 55+"
2,customers.csv,city,object,121930,0,42,"Hai Phong, Hai Phong, Hai Phong, Hai Phong, Hai Phong"
3,customers.csv,customer_id,int64,121930,0,121930,"1, 2, 3, 4, 5"
4,customers.csv,gender,object,121930,0,3,"Female, Female, Female, Male, Male"
5,customers.csv,signup_date,object,121930,0,3941,"2021-12-30, 2013-12-27, 2018-07-24, 2017-11-29, 2022-09-23"
6,customers.csv,zip,int64,121930,0,31491,"15201, 15201, 15201, 15201, 15201"
7,geography.csv,city,object,39948,0,42,"Hai Phong, Phu Ly, Viet Tri, Bac Giang, Bac Giang"
8,geography.csv,district,object,39948,0,39,"District #13, District #13, District #13, District #13, District #13"
9,geography.csv,region,object,39948,0,3,"East, East, East, East, East"


## 5. Missing values summary

Report missing count and missing percentage for every column. The table is sorted so the most missing columns are easiest to inspect.

In [5]:
missing_rows = []
for filename, df in loaded.items():
    total_rows = len(df)
    for col in df.columns:
        missing_count = int(df[col].isna().sum())
        missing_rows.append({
            "file": filename,
            "column": col,
            "missing_count": missing_count,
            "missing_pct": (missing_count / total_rows * 100) if total_rows else 0.0,
        })

missing_summary_df = pd.DataFrame(missing_rows).sort_values(
    ["missing_pct", "missing_count", "file", "column"], ascending=[False, False, True, True]
).reset_index(drop=True)
missing_summary_df

,file,column,missing_count,missing_pct
0,order_items.csv,promo_id_2,714463,99.971175
1,promotions.csv,applicable_category,40,80.000000
2,order_items.csv,promo_id,438353,61.336507
3,customers.csv,acquisition_channel,0,0.000000
4,customers.csv,age_group,0,0.000000
5,customers.csv,city,0,0.000000
6,customers.csv,customer_id,0,0.000000
7,customers.csv,gender,0,0.000000
8,customers.csv,signup_date,0,0.000000
9,customers.csv,zip,0,0.000000


## 6. Duplicate rows summary

Count fully duplicated rows in each dataset.

In [6]:
duplicate_rows = []
for filename, df in loaded.items():
    duplicate_count = int(df.duplicated().sum())
    duplicate_rows.append({
        "file": filename,
        "rows": len(df),
        "duplicate_rows": duplicate_count,
        "duplicate_pct": (duplicate_count / len(df) * 100) if len(df) else 0.0,
    })

duplicate_summary_df = pd.DataFrame(duplicate_rows).sort_values(
    ["duplicate_rows", "file"], ascending=[False, True]
).reset_index(drop=True)
duplicate_summary_df

,file,rows,duplicate_rows,duplicate_pct
0,customers.csv,121930,0,0.0
1,geography.csv,39948,0,0.0
2,inventory.csv,60247,0,0.0
3,order_items.csv,714669,0,0.0
4,orders.csv,646945,0,0.0
5,payments.csv,646945,0,0.0
6,products.csv,2412,0,0.0
7,promotions.csv,50,0,0.0
8,returns.csv,39939,0,0.0
9,reviews.csv,113551,0,0.0


## 7. Date column detection and date ranges

Detect date-like columns by column name and by parse success rate. For each detected column, report whether pandas loaded it as datetime, parse success rate, and min/max parsed date.

In [7]:
def parse_datetime_series(series):
    return pd.to_datetime(series, errors="coerce")


def date_detection_reason(column_name, series):
    name_match = bool(DATE_NAME_PATTERN.search(str(column_name)))
    dtype_datetime = pd.api.types.is_datetime64_any_dtype(series)

    parse_success_rate = np.nan
    parsed = pd.Series(pd.NaT, index=series.index, dtype="datetime64[ns]")
    non_null = series.dropna()

    # Avoid treating ordinary numeric identifiers as timestamps by parse-rate alone.
    should_try_parse = name_match or dtype_datetime or pd.api.types.is_object_dtype(series) or pd.api.types.is_string_dtype(series)
    if len(non_null) > 0 and should_try_parse:
        parsed = parse_datetime_series(series)
        parse_success_rate = float(parsed.notna().sum() / len(non_null))
    elif len(non_null) == 0:
        parse_success_rate = np.nan

    high_parse = bool(pd.notna(parse_success_rate) and parse_success_rate >= DATE_PARSE_THRESHOLD)
    detected = name_match or dtype_datetime or high_parse

    reasons = []
    if name_match:
        reasons.append("name_match")
    if dtype_datetime:
        reasons.append("datetime_dtype")
    if high_parse:
        reasons.append("high_parse_success")

    return detected, ";".join(reasons), parse_success_rate, parsed


date_rows = []
for filename, df in loaded.items():
    for col in df.columns:
        detected, reason, parse_success_rate, parsed = date_detection_reason(col, df[col])
        if not detected:
            continue
        parsed_non_null = parsed.dropna()
        date_rows.append({
            "file": filename,
            "column": col,
            "dtype_loaded": str(df[col].dtype),
            "loaded_as_datetime": bool(pd.api.types.is_datetime64_any_dtype(df[col])),
            "detection_reason": reason,
            "non_null_count": int(df[col].notna().sum()),
            "parse_success_rate": parse_success_rate,
            "min_parsed_date": parsed_non_null.min() if len(parsed_non_null) else pd.NaT,
            "max_parsed_date": parsed_non_null.max() if len(parsed_non_null) else pd.NaT,
        })

date_summary_df = pd.DataFrame(date_rows).sort_values(["file", "column"]).reset_index(drop=True)
date_summary_df

,file,column,dtype_loaded,loaded_as_datetime,detection_reason,non_null_count,parse_success_rate,min_parsed_date,max_parsed_date
0,customers.csv,signup_date,object,False,name_match;high_parse_success,121930,1.0,2012-01-17,2022-12-31
1,inventory.csv,snapshot_date,object,False,name_match;high_parse_success,60247,1.0,2012-07-31,2022-12-31
2,orders.csv,order_date,object,False,name_match;high_parse_success,646945,1.0,2012-07-04,2022-12-31
3,promotions.csv,end_date,object,False,name_match;high_parse_success,50,1.0,2013-03-01,2022-12-31
4,promotions.csv,start_date,object,False,name_match;high_parse_success,50,1.0,2013-01-31,2022-11-18
5,returns.csv,return_date,object,False,name_match;high_parse_success,39939,1.0,2012-07-11,2022-12-31
6,reviews.csv,review_date,object,False,name_match;high_parse_success,113551,1.0,2012-07-10,2022-12-31
7,sales.csv,Date,object,False,name_match;high_parse_success,3833,1.0,2012-07-04,2022-12-31
8,sample_submission.csv,Date,object,False,name_match;high_parse_success,548,1.0,2023-01-01,2024-07-01
9,shipments.csv,delivery_date,object,False,name_match;high_parse_success,566067,1.0,2012-07-06,2022-12-31


## 8. Potential numeric columns read as strings

For object/string columns, attempt numeric conversion and flag columns where most non-null values behave like numbers.

In [8]:
def numeric_parse_success(series):
    non_null = series.dropna()
    if len(non_null) == 0:
        return np.nan, pd.Series(dtype="float64")
    cleaned = (
        non_null.astype(str)
        .str.strip()
        .str.replace(",", "", regex=False)
        .str.replace("%", "", regex=False)
    )
    parsed = pd.to_numeric(cleaned, errors="coerce")
    return float(parsed.notna().sum() / len(non_null)), parsed

numeric_candidate_rows = []
for filename, df in loaded.items():
    for col in df.columns:
        if not (pd.api.types.is_object_dtype(df[col]) or pd.api.types.is_string_dtype(df[col])):
            continue
        success_rate, parsed = numeric_parse_success(df[col])
        if pd.notna(success_rate) and success_rate >= NUMERIC_PARSE_THRESHOLD:
            numeric_candidate_rows.append({
                "file": filename,
                "column": col,
                "dtype_loaded": str(df[col].dtype),
                "non_null_count": int(df[col].notna().sum()),
                "numeric_parse_success_rate": success_rate,
                "parsed_min": parsed.min() if len(parsed.dropna()) else np.nan,
                "parsed_max": parsed.max() if len(parsed.dropna()) else np.nan,
                "sample_values": ", ".join(df[col].dropna().astype(str).head(5).tolist()),
            })

numeric_string_columns = [
    "file",
    "column",
    "dtype_loaded",
    "non_null_count",
    "numeric_parse_success_rate",
    "parsed_min",
    "parsed_max",
    "sample_values",
]
numeric_string_summary_df = pd.DataFrame(numeric_candidate_rows, columns=numeric_string_columns)
if len(numeric_string_summary_df):
    numeric_string_summary_df = numeric_string_summary_df.sort_values(
        ["file", "column"]
    ).reset_index(drop=True)
numeric_string_summary_df

,file,column,dtype_loaded,non_null_count,numeric_parse_success_rate,parsed_min,parsed_max,sample_values


## 9. Key observations / warnings

Produce concise warning tables and a printed summary for quick review.

In [9]:
high_missing_df = missing_summary_df[missing_summary_df["missing_pct"] >= HIGH_MISSING_THRESHOLD].copy()
duplicate_warning_df = duplicate_summary_df[duplicate_summary_df["duplicate_rows"] > 0].copy()
low_date_parse_df = date_summary_df[
    date_summary_df["parse_success_rate"].notna()
    & (date_summary_df["parse_success_rate"] < DATE_PARSE_THRESHOLD)
].copy()

print("Warning summary")
print("- Missing files:", missing_files_df["file"].tolist() if len(missing_files_df) else "None")
print(
    "- Datasets/columns with high missingness:",
    high_missing_df[["file", "column", "missing_pct"]].to_dict("records") if len(high_missing_df) else "None",
)
print(
    "- Datasets with duplicate rows:",
    duplicate_warning_df[["file", "duplicate_rows", "duplicate_pct"]].to_dict("records") if len(duplicate_warning_df) else "None",
)
print(
    "- Date columns with low parse success:",
    low_date_parse_df[["file", "column", "parse_success_rate"]].to_dict("records") if len(low_date_parse_df) else "None",
)
print(
    "- Columns likely numeric stored as string:",
    numeric_string_summary_df[["file", "column", "numeric_parse_success_rate"]].to_dict("records") if len(numeric_string_summary_df) else "None",
)

print("\nMissing files summary")
display(missing_files_df if len(missing_files_df) else pd.DataFrame({"message": ["No expected files are missing."]}))

print("\nHigh missingness columns")
display(high_missing_df if len(high_missing_df) else pd.DataFrame({"message": ["No columns exceed the high-missingness threshold."]}))

print("\nDuplicate row warnings")
display(duplicate_warning_df if len(duplicate_warning_df) else pd.DataFrame({"message": ["No fully duplicated rows detected."]}))

print("\nLow date parse success warnings")
display(low_date_parse_df if len(low_date_parse_df) else pd.DataFrame({"message": ["No detected date columns have low parse success."]}))

print("\nPotential numeric-as-string warnings")
display(numeric_string_summary_df if len(numeric_string_summary_df) else pd.DataFrame({"message": ["No object/string columns appear to be numeric."]}))

Warning summary
- Missing files: None
- Datasets/columns with high missingness: [{'file': 'order_items.csv', 'column': 'promo_id_2', 'missing_pct': 99.97117546724428}, {'file': 'promotions.csv', 'column': 'applicable_category', 'missing_pct': 80.0}, {'file': 'order_items.csv', 'column': 'promo_id', 'missing_pct': 61.336506830434786}]
- Datasets with duplicate rows: None
- Date columns with low parse success: None
- Columns likely numeric stored as string: None

Missing files summary


,message
0,No expected files are missing.



High missingness columns


,file,column,missing_count,missing_pct
0,order_items.csv,promo_id_2,714463,99.971175
1,promotions.csv,applicable_category,40,80.000000
2,order_items.csv,promo_id,438353,61.336507



Duplicate row warnings


,message
0,No fully duplicated rows detected.



Low date parse success warnings


,message
0,No detected date columns have low parse success.



Potential numeric-as-string warnings


,message
0,No object/string columns appear to be numeric.


## 10. Missing value treatment recommendations

### `order_items.csv.promo_id` - 61.34% missing

This likely means most order items had no promotion, not bad data.

Recommended treatment:

```text
promo_id missing
= no primary promotion applied
```

Feature engineering:

```python
has_promo = promo_id.notna()
promo_id_filled = promo_id.fillna("NO_PROMO")
```

For modeling, use:

- `has_promo`
- count/share of promoted items per day
- total discount amount per day
- promotion type/channel after joining `promotions.csv`

Do not drop these rows.

### `order_items.csv.promo_id_2` - 99.97% missing

This likely means a second stacked promotion was almost never used.

Recommended treatment:

```text
promo_id_2 missing
= no secondary promotion applied
```

Feature engineering:

```python
has_second_promo = promo_id_2.notna()
promo_id_2_filled = promo_id_2.fillna("NO_SECOND_PROMO")
```

Because it is almost always missing, avoid using raw `promo_id_2` as a high-cardinality categorical feature. A simple binary flag is probably enough:

- `has_second_promo`

You may also aggregate daily:

- `second_promo_item_count`
- `second_promo_item_share`

### `promotions.csv.applicable_category` - 80% missing

This likely means the promotion applies to all categories, not unknown.

Recommended treatment:

```text
applicable_category missing
= all categories / no category restriction
```

Feature engineering:

```python
applicable_category_filled = applicable_category.fillna("ALL_CATEGORIES")
is_category_specific = applicable_category.notna()
```

For modeling, useful features include:

- active promo count by date
- active category-specific promo flag
- active all-category promo flag
- discount value
- promo channel
- promo type
- stackable flag
- min order value

Do not impute this with the mode. The missingness itself carries meaning.
